# Week 2: Baseline modelling — Viscosity prediction (Expanded Formulations)

Predicting viscosity at different shear rates using baseline regression models.

**Targets:**
1. `Viscosity_at_shear_rate_1_1/s`
2. `Viscosity_at_shear_rate_10_1/s`
3. `Viscosity_at_shear_rate_100_1/s`
4. `Viscosity_avg` (mean of the three)

**Data split:** Group-aware splitting using `Composite_Mix_ID` to prevent experiment replicates from appearing in both train and test sets.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# --- Config ---
VISCOSITY_COLS = [
    "Viscosity_at_shear_rate_1_1/s",
    "Viscosity_at_shear_rate_10_1/s",
    "Viscosity_at_shear_rate_100_1/s",
]
TEST_SIZE = 0.25
RANDOM_STATE = 42

DATA_PATH = Path("../data/processed/combined_slurry_data_expanded.csv")

In [3]:
df = pd.read_csv(DATA_PATH)

# Extract group column for group-aware splitting (experiment replicates)
groups = df["Composite_Mix_ID"]
df = df.drop(columns=["Composite_Mix_ID"])

# Create averaged viscosity target
df["Viscosity_avg"] = df[VISCOSITY_COLS].mean(axis=1)

# All viscosity targets (individual + averaged)
all_targets = VISCOSITY_COLS + ["Viscosity_avg"]

# Feature columns (everything except the viscosity targets)
feature_cols = [c for c in df.columns if c not in all_targets]
X_full = df[feature_cols]

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()

print(f"Rows: {len(df)}")
print(f"Unique experiment groups: {groups.nunique()}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"  Numeric: {num_cols}")
print(f"  Categorical: {cat_cols}")
print(f"\nTargets to predict: {all_targets}")

Rows: 178
Unique experiment groups: 68
Features (8): ['Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'Source_Batch', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct']
  Numeric: ['Solid_Content_pct', 'Solid_Additive_pct', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct']
  Categorical: ['Dispersent_Type', 'Source_Batch']

Targets to predict: ['Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s', 'Viscosity_avg']


In [4]:
# Preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

# Baseline models
models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso": Lasso(alpha=0.1, max_iter=5000, random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "DecisionTree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

# Group-aware train/test split (keeps experiment replicates together)
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_full, groups=groups))

print(f"Train: {len(train_idx)} samples ({len(set(groups.iloc[train_idx]))} groups)")
print(f"Test:  {len(test_idx)} samples ({len(set(groups.iloc[test_idx]))} groups)")

# Verify no group leakage
assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0, "Group leakage detected!"
print("✅ No group leakage - experiment groups are mutually exclusive\n")

# Train and evaluate models
all_results = []

for target in all_targets:
    y = df[target]
    X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    for name, model in models.items():
        pipe = Pipeline([
            ("preprocess", preprocess),
            ("model", model),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        
        all_results.append({
            "target": target.replace("Viscosity_at_shear_rate_", "SR_").replace("1/s", ""),
            "model": name,
            "R²": r2,
            "RMSE": rmse,
            "MAE": mae,
        })

results_df = pd.DataFrame(all_results)
results_df

Train: 136 samples (51 groups)
Test:  42 samples (17 groups)
✅ No group leakage - experiment groups are mutually exclusive



,target,model,R²,RMSE,MAE
0,SR_1_,Dummy (mean),-0.079377,85.430719,62.851420
1,SR_1_,Ridge,0.589680,52.673109,35.914686
2,SR_1_,Lasso,0.589581,52.679427,35.932073
3,SR_1_,KNN (k=5),0.444335,61.296278,34.178724
4,SR_1_,DecisionTree,0.653637,48.394197,31.078490
5,SR_1_,RandomForest,0.695272,45.392482,29.280480
6,SR_10_,Dummy (mean),-0.088826,14.983403,11.373661
7,SR_10_,Ridge,0.749574,7.185738,5.108791
8,SR_10_,Lasso,0.739181,7.333318,5.232421
9,SR_10_,KNN (k=5),0.642595,8.584433,5.300710


In [5]:
print("=" * 60)
print("BASELINE REGRESSION RESULTS (sorted by R² per target)")
print("=" * 60)

for target in results_df["target"].unique():
    subset = results_df[results_df["target"] == target].sort_values("R²", ascending=False)
    print(f"\n▶ {target}")
    display(subset.style.format({"R²": "{:.3f}", "RMSE": "{:.2f}", "MAE": "{:.2f}"}).hide(axis="index"))

# Summary: best model per target
print("\n" + "=" * 60)
print("BEST MODEL PER TARGET (by R²)")
print("=" * 60)
best_per_target = results_df.loc[results_df.groupby("target")["R²"].idxmax()]
display(best_per_target.style.format({"R²": "{:.3f}", "RMSE": "{:.2f}", "MAE": "{:.2f}"}).hide(axis="index"))

BASELINE REGRESSION RESULTS (sorted by R² per target)

▶ SR_1_


target,model,R²,RMSE,MAE
SR_1_,RandomForest,0.695,45.39,29.28
SR_1_,DecisionTree,0.654,48.39,31.08
SR_1_,Ridge,0.590,52.67,35.91
SR_1_,Lasso,0.590,52.68,35.93
SR_1_,KNN (k=5),0.444,61.30,34.18
SR_1_,Dummy (mean),-0.079,85.43,62.85



▶ SR_10_


target,model,R²,RMSE,MAE
SR_10_,RandomForest,0.839,5.76,3.35
SR_10_,DecisionTree,0.766,6.94,4.31
SR_10_,Ridge,0.750,7.19,5.11
SR_10_,Lasso,0.739,7.33,5.23
SR_10_,KNN (k=5),0.643,8.58,5.30
SR_10_,Dummy (mean),-0.089,14.98,11.37



▶ SR_100_


target,model,R²,RMSE,MAE
SR_100_,RandomForest,0.915,1.11,0.69
SR_100_,DecisionTree,0.903,1.19,0.86
SR_100_,Ridge,0.870,1.38,1.02
SR_100_,Lasso,0.844,1.51,1.11
SR_100_,KNN (k=5),0.746,1.93,1.34
SR_100_,Dummy (mean),-0.078,3.98,3.27



▶ Viscosity_avg


target,model,R²,RMSE,MAE
Viscosity_avg,RandomForest,0.739,16.86,10.94
Viscosity_avg,DecisionTree,0.707,17.88,11.74
Viscosity_avg,Ridge,0.634,19.98,13.84
Viscosity_avg,Lasso,0.633,20.00,13.81
Viscosity_avg,KNN (k=5),0.488,23.62,13.39
Viscosity_avg,Dummy (mean),-0.083,34.36,25.66



BEST MODEL PER TARGET (by R²)


target,model,R²,RMSE,MAE
SR_100_,RandomForest,0.915,1.11,0.69
SR_10_,RandomForest,0.839,5.76,3.35
SR_1_,RandomForest,0.695,45.39,29.28
Viscosity_avg,RandomForest,0.739,16.86,10.94


## Data Leakage Audit

In [6]:
print("🔍 DATA LEAKAGE AUDIT")
print("=" * 60)

# 1. Check: Are any target columns in the features?
target_in_features = [c for c in all_targets if c in feature_cols]
if target_in_features:
    print(f"❌ LEAKAGE: Target columns in features: {target_in_features}")
else:
    print("✅ No target columns in features")

# 2. Check: Group-aware splitting
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])
leaked_groups = train_groups & test_groups
if leaked_groups:
    print(f"❌ GROUP LEAKAGE: {len(leaked_groups)} groups in both train and test")
else:
    print(f"✅ Group-aware split: {len(train_groups)} train groups, {len(test_groups)} test groups (no overlap)")

# 3. Check: Replicate distribution
print(f"\n📊 Replicate distribution (samples per experiment group):")
display(groups.value_counts().value_counts().sort_index().to_frame("# of groups"))

# 4. Check: Viscosity column correlations
print("\n📊 Viscosity column correlations:")
visc_corr = df[VISCOSITY_COLS].corr()
display(visc_corr.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

# 5. Check: Feature-target correlations
print("\n📈 Feature-Target correlations (|r| > 0.7 flagged):")
numeric_features = X_full.select_dtypes(include=[np.number]).columns
for target in all_targets:
    short_name = target.replace("Viscosity_at_shear_rate_", "SR_").replace("1/s", "")
    high_corr = []
    for feat in numeric_features:
        corr = df[feat].corr(df[target])
        if abs(corr) > 0.7:
            high_corr.append((feat, round(corr, 3)))
    if high_corr:
        print(f"   ⚠️  {short_name}: {high_corr} (legitimate physical relationship)")
    else:
        print(f"   ✅ {short_name}: No high correlations")

# 6. Pipeline check
print("\n🔄 Pipeline Check:")
print("   • ColumnTransformer inside Pipeline: ✅ (fit only on train data)")
print("   • GroupShuffleSplit used: ✅ (experiment replicates kept together)")

🔍 DATA LEAKAGE AUDIT
✅ No target columns in features
✅ Group-aware split: 51 train groups, 17 test groups (no overlap)

📊 Replicate distribution (samples per experiment group):


,# of groups
count,
1,17
2,20
3,15
4,4
5,12



📊 Viscosity column correlations:


,Viscosity_at_shear_rate_1_1/s,Viscosity_at_shear_rate_10_1/s,Viscosity_at_shear_rate_100_1/s
Viscosity_at_shear_rate_1_1/s,1.000,0.921,0.810
Viscosity_at_shear_rate_10_1/s,0.921,1.000,0.961
Viscosity_at_shear_rate_100_1/s,0.810,0.961,1.000



📈 Feature-Target correlations (|r| > 0.7 flagged):
   ✅ SR_1_: No high correlations
   ⚠️  SR_10_: [('Solid_Content_pct', np.float64(0.801))] (legitimate physical relationship)
   ⚠️  SR_100_: [('Solid_Content_pct', np.float64(0.858))] (legitimate physical relationship)
   ⚠️  Viscosity_avg: [('Solid_Content_pct', np.float64(0.72))] (legitimate physical relationship)

🔄 Pipeline Check:
   • ColumnTransformer inside Pipeline: ✅ (fit only on train data)
   • GroupShuffleSplit used: ✅ (experiment replicates kept together)
